# Notebook 02: Disagreement Categorisation

**Question:** What do the disagreement inputs have in common? Is there a pattern?

**Method:** Extract text features (length, vocabulary rarity, named entities, negation) 
from all test inputs. Compare feature distributions between disagreement inputs 
and agreement inputs using statistical tests (KS test for continuous features).

**Focus:** Full model vs ONNX INT8 — the only comparison with disagreements.

In [ ]:
import sys
sys.path.append('..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from quantscope import (
    load_predictions, build_training_vocab,
    categorise_all, compare_categories, print_category_report
)

sns.set_theme(style='whitegrid')

In [ ]:
import os
os.makedirs('../results/charts', exist_ok=True)
print('✅ charts/ folder created')

In [ ]:
# Load predictions and disagreement summary
data = load_predictions('../results/all_predictions.json')

with open('../results/disagreement_summary.json') as f:
    disagreement = json.load(f)

texts  = data['texts']
labels = data['labels']

# We focus on full vs int8 — fp32 had zero disagreements
hurt_indices     = disagreement['full_vs_int8']['hurt_indices']
helped_indices   = disagreement['full_vs_int8']['helped_indices']
disagree_indices = disagreement['full_vs_int8']['disagreement_indices']

print(f'Total samples       : {len(texts)}')
print(f'Disagreements       : {len(disagree_indices)}')
print(f'Hurt (full better)  : {len(hurt_indices)}')
print(f'Helped (int8 better): {len(helped_indices)}')

In [ ]:
# Build vocabulary from texts to measure rarity
print('Building vocabulary...')
vocab = build_training_vocab(texts, min_count=3)
print(f'Vocab size: {len(vocab)} tokens')

In [ ]:
# Extract features from all texts
features = categorise_all(texts, training_vocab=vocab)
print('✅ Feature extraction complete')
print(f'Sample features: {features[0]}')

In [ ]:
# Compare feature distributions: disagreement vs agreement inputs
comparison = compare_categories(features, disagree_indices)
print_category_report(comparison)

In [ ]:
# Chart 1: Disagreement by category
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Feature Distributions: Disagreement vs Agreement Inputs', fontsize=13)

w = 0.35

# Length buckets
length_data = {
    'Short (<50)'    : [comparison['length_short']['disagree_pct'],  comparison['length_short']['agree_pct']],
    'Medium (50-150)': [comparison['length_medium']['disagree_pct'], comparison['length_medium']['agree_pct']],
    'Long (>150)'    : [comparison['length_long']['disagree_pct'],   comparison['length_long']['agree_pct']],
}
x = np.arange(len(length_data))
axes[0].bar(x - w/2, [v[0] for v in length_data.values()], w, label='Disagreement', color='#ef4444')
axes[0].bar(x + w/2, [v[1] for v in length_data.values()], w, label='Agreement',    color='#3b82f6')
axes[0].set_xticks(x)
axes[0].set_xticklabels(length_data.keys(), rotation=15)
axes[0].set_ylabel('% of inputs')
axes[0].set_title('Text Length')
axes[0].legend()

# Negation
neg_data = {
    'Has Negation': [comparison['negation']['disagree_pct'], comparison['negation']['agree_pct']],
    'No Negation' : [100 - comparison['negation']['disagree_pct'], 100 - comparison['negation']['agree_pct']],
}
x2 = np.arange(len(neg_data))
axes[1].bar(x2 - w/2, [v[0] for v in neg_data.values()], w, label='Disagreement', color='#ef4444')
axes[1].bar(x2 + w/2, [v[1] for v in neg_data.values()], w, label='Agreement',    color='#3b82f6')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(neg_data.keys())
axes[1].set_title('Negation Presence')
axes[1].legend()

# Entity presence
ent_data = {
    'Has Entities': [comparison['has_entities']['disagree_pct'], comparison['has_entities']['agree_pct']],
    'No Entities' : [100 - comparison['has_entities']['disagree_pct'], 100 - comparison['has_entities']['agree_pct']],
}
x3 = np.arange(len(ent_data))
axes[2].bar(x3 - w/2, [v[0] for v in ent_data.values()], w, label='Disagreement', color='#ef4444')
axes[2].bar(x3 + w/2, [v[1] for v in ent_data.values()], w, label='Agreement',    color='#3b82f6')
axes[2].set_xticks(x3)
axes[2].set_xticklabels(ent_data.keys())
axes[2].set_title('Named Entity Presence')
axes[2].legend()

plt.tight_layout()
plt.savefig('../results/charts/disagreement_by_category.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved disagreement_by_category.png')

In [ ]:
# Inspect hurt cases — what do they look like?
print('=== HURT CASES (full correct, INT8 wrong) ===\n')
for idx in hurt_indices[:10]:
    f = features[idx]
    print(f"[{idx}] Label={labels[idx]} | len={f['char_len']} | "
          f"rarity={f['rarity_pct']} | entities={f['n_entities']} | "
          f"negation={f['has_negation']}")
    print(f"  Text: {texts[idx][:120]}")
    print()

In [ ]:
# Save category comparison for README
with open('../results/category_comparison.json', 'w') as f:
    json.dump(comparison, f, indent=2)
print('✅ Saved category_comparison.json')